# Sprint 1 · Presentación del dataset e ingesta

**Tesis:** Diagnóstico de causalidad de fallas en una red telco usando grafos · Región **Pasco**, Perú.

Este notebook **presenta el dataset** y **prueba el pipeline de ingesta** del Sprint 1. La lógica vive en
`src/ingesta.py` (módulo reutilizable); aquí solo lo **importamos y ejecutamos** — así el mismo código se usa
después en el EDA y en las fases de grafo/motor, sin duplicarlo.

| Entregable de la tarjeta | Dónde |
|---|---|
| Comprensión del negocio | `docs/01_comprension_negocio.md` |
| Diccionario de datos | `docs/02_diccionario_datos.md` |
| **Script reproducible de ingesta** | **`src/ingesta.py` (ejecutado aquí)** |


## 1. Presentación del dataset

Los datos se descargan de los **gestores Huawei NCE** por *web scraping*, cada **15 minutos**. Cada descarga
es una **ventana** (carpeta `DD_MM_AAAA_HH_MM_SS`) y constituye la **unidad de análisis**. Dentro de cada ventana
hay 6 archivos `.json` (3 de **estado** de equipos + 3 de **alarmas**, para transporte, acceso-router y
acceso-radio) y, junto a las ventanas, el inventario estático **`red.xlsx`** (hoja `nodos`).

- **Datos estructurados:** estado de equipos, severidad, timestamps, IP, inventario de nodos.
- **Datos semi/no estructurados:** `moi` (codifica subrack/placa/puerto), `alarmName`, `probableCause`.

> Nota de seguridad: los scripts de scraping usan credenciales privadas del operador y **no** forman parte del
> repositorio. Solo se versiona el **código de ingesta**; los `.json` crudos quedan en `data/raw/` (ignorados por git).

## 2. Configuración (rutas locales + import del módulo)

Se añade la raíz del repo a `sys.path` para poder `from src.ingesta import ...` aunque el notebook esté en
`notebooks/`. Ajusta `BASE_DIR` a la carpeta local donde tengas las ventanas y `red.xlsx`.

In [ ]:
import os, sys
from pathlib import Path
import pandas as pd

# --- raíz del repo (para importar src/) ---
ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# --- ruta a los datos crudos (ajústala a tu máquina) ---
BASE_DIR = str(ROOT / "data" / "raw")
RED_XLSX = os.path.join(BASE_DIR, "red.xlsx")

assert os.path.isdir(BASE_DIR), f"No se encontró la carpeta de datos: {BASE_DIR}"
print("Repo :", ROOT)
print("Datos:", BASE_DIR)

# vista rápida del inventario
red_preview = pd.read_excel(RED_XLSX, sheet_name="nodos")
print("red.xlsx (nodos):", red_preview.shape, "| columnas:", list(red_preview.columns))
red_preview.head()

## 3. Diagrama de conexión de tablas

Cómo se unen las fuentes dentro de una ventana (estado→alarmas por IP; luego inventario por `IP_Equipo = address`).

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(11, 5)); ax.axis("off")
def caja(x, y, w, h, t, col):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.03",
                 fc=col, ec="#333", lw=1.2))
    ax.text(x + w/2, y + h/2, t, ha="center", va="center", fontsize=9)
def flecha(x1, y1, x2, y2, t=""):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle="->",
                 mutation_scale=16, lw=1.3, color="#444"))
    if t: ax.text((x1+x2)/2, (y1+y2)/2 + 0.12, t, ha="center", fontsize=8, color="#444")

caja(0.3, 3.4, 2.6, 0.9, "ESTADO equipos\n(3 .json)", "#dbeafe")
caja(0.3, 1.6, 2.6, 0.9, "ALARMAS\n(3 .json)", "#fde68a")
caja(4.2, 2.5, 2.6, 0.9, "Alarmas + estado\n(por IP)", "#bbf7d0")
caja(8.0, 2.5, 2.7, 0.9, "Dataset consolidado\n+ inventario (red.xlsx)", "#c7d2fe")
caja(8.0, 0.6, 2.7, 0.8, "red.xlsx (nodos)", "#e5e7eb")
flecha(2.9, 3.6, 4.2, 3.1, "address = NE IP")
flecha(2.9, 2.0, 4.2, 2.8)
flecha(6.8, 2.95, 8.0, 2.95, "+ ventana_id/ts")
flecha(9.35, 1.4, 9.35, 2.5, "IP_Equipo = address")
ax.set_xlim(0, 11); ax.set_ylim(0, 5)
plt.tight_layout(); plt.show()

## 4. Ingesta — ejecución del pipeline (`src/ingesta.py`)

`construir_dataset` recorre **todas** las ventanas de `BASE_DIR`, une estado→alarmas, apila las 3 redes,
anexa el inventario (con su `Red` y el `rol_equipo`) y marca `ventana_id`/`ventana_ts`. Aplica dos filtros de
alcance: (1) solo equipos que están en `red.xlsx`, y (2) separa los **sectoriales (RTN510_AP)**, que no afectan
la conectividad del nodo y se analizan aparte como indicador de clientes.

In [ ]:
from src.ingesta import (construir_dataset, parsear_moi, marcar_particion_temporal,
                         indicador_sectoriales)

df, df_sect = construir_dataset(BASE_DIR, devolver_sectoriales=True)   # procesa TODAS las ventanas
print("\nConsolidado (causa raíz):", df.shape)
print("Ventanas:", df["ventana_id"].nunique(),
      "| rango:", df["ventana_ts"].min(), "→", df["ventana_ts"].max())
df.head()

> **Resumen.** Los mensajes de arriba informan cada filtro: equipos del inventario con IP no válida (`PENDIENTE`), equipos sin código de nodo, alarmas fuera de inventario descartadas y alarmas de sectoriales separadas. La tabla muestra las primeras filas del dataset que sí entra al diagnóstico de causa raíz.

In [ ]:
# Listado de columnas y un vistazo rápido por red / rol de equipo / severidad
print("Columnas (", df.shape[1], "):")
print(list(df.columns))
print()
for col in ["Red", "rol_equipo", "segmento_red", "severidad_txt"]:
    if col in df.columns:
        print(df[col].value_counts(dropna=False), "\n")

> **Resumen.** Lista de columnas del consolidado y cómo se reparten las alarmas por **red** (tx/ax), por **rol de equipo** (router, microondas, transporte…), por segmento y por severidad.

### 4.1 Indicador de sectoriales (salud de clientes)

Las alarmas de los sectoriales **no** entran al diagnóstico de causa raíz, pero sirven como indicador: cuántos
sectoriales tienen alarmas en cada ventana (p. ej. alarmas de *cliente no conectado*).

In [ ]:
ind = indicador_sectoriales(df_sect)
print("Alarmas de sectoriales:", len(df_sect), "| sectoriales distintos:", df_sect["address"].nunique() if len(df_sect) else 0)
if len(df_sect):
    print("\nTipos de alarma más frecuentes en sectoriales:")
    display(df_sect["alarmName"].value_counts().head(10).to_frame("n"))
ind.head()

> **Qué muestran las tablas.** El ranking de **tipos de alarma en los sectoriales** (útil para identificar la alarma de *cliente no conectado*) y, por ventana, **cuántos sectoriales tienen alarma**, cuántas alarmas suman y en cuántos nodos están. Es un indicador separado del diagnóstico de causa raíz.

## 5. Pre-procesamiento

### 5.1 Anclaje a puerto (parsing de `moi`)

`parsear_moi` extrae, cuando existen, `moi_subrack` / `moi_placa` / `moi_puerto` del campo semiestructurado.
Las alarmas de gestión (OSS) y de RTN sin puerto quedan en `NaN` — es esperado y queda documentado.

In [ ]:
df = parsear_moi(df)
cobertura = df["moi_puerto"].notna().mean()
print(f"Alarmas con puerto identificado: {cobertura:.1%}")
df.loc[df["moi_puerto"].notna(), ["moi", "moi_subrack", "moi_placa", "moi_puerto"]].head(8)

### 5.2 Partición temporal (train/test)

Partimos **por tiempo a nivel de ventana** (las más antiguas → train) para no filtrar información del futuro
hacia el pasado. Queda en la columna `split`.

In [ ]:
df = marcar_particion_temporal(df, frac_train=0.8)
print(df.groupby("split")["ventana_id"].nunique().rename("n_ventanas"))
print()
print(df["split"].value_counts().rename("n_filas"))

## 6. Guardar el dataset consolidado

Se guarda en `data/processed/` en CSV (revisión rápida), JSON (intercambio) y Parquet (eficiente para el EDA).

In [ ]:
out = ROOT / "data" / "processed"
out.mkdir(parents=True, exist_ok=True)
df.to_csv(out / "dataset_consolidado.csv", index=False, encoding="utf-8-sig")
df.to_json(out / "dataset_consolidado.json", orient="records",
           force_ascii=False, date_format="iso", indent=2)
try:
    df.to_parquet(out / "dataset_consolidado.parquet", index=False)
except Exception as e:
    print("Parquet omitido (instala pyarrow):", e)
# sectoriales: dataset aparte + indicador por ventana
if len(df_sect):
    df_sect.to_csv(out / "alarmas_sectoriales.csv", index=False, encoding="utf-8-sig")
    ind.to_csv(out / "sectoriales_por_ventana.csv", index=False, encoding="utf-8-sig")
print("Guardado en", out)